# Visualize Rates
### Import libraries

In [1]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm
%matplotlib qt

In [2]:
from utils import add_package_path

Paths

In [3]:
# Current Working Directory
cwd = os.getcwd()
# WORKSPACE
ws = os.environ['WORKSPACE_PATH']

In [4]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET'
paths = {
    # PyCOMET source dir
    'pycomet': prefix + '/pycomet/',
    # SurQCT source dir
    'surqct': prefix + '/pycomet/examples/SurQCT/'
}
add_package_path(paths)

In [5]:
# Output directory
out = cwd + '/Output/VisualizeRates/Inel/final/c01.ref/'
if not os.path.exists(out):
    os.makedirs(out)

### Inputs

In [6]:
SystemName   = 'O3_UMN'
RatesId      = 'Exch_1'
TTranVec     = [5000]
MinRateValue = 1.e-16
# iIdxVec      = np.arange(10)
iIdxVec      = np.random.randint(low=0, high=6114, size=10)

SuffixesHDF5 = {
    # 'QCT':    '/Orig/',
    'SurQCT-ref': '/final/ref/',
    'SurQCT': '/excit.qct_diss/c01.ref/'
    # 'SurQCT': '/final/ref/'
}

### Initialize SurQCT objects

Import additional modules

In [7]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import RatesType
from surqct import config as cfg

Initializing QCT & SurQCT chemical systems

In [8]:
Systems = {}
for Identifier, SuffixHDF5 in SuffixesHDF5.items():
    KinSource = Identifier.split('-')[0]
    # Initializing rates
    Systems[Identifier] = syst.get_system(
        NameLong=SystemName,
        KinSource=KinSource,
        ProcNames=tf.nest.flatten(RatesId),
        PathToHDF5Fld=cfg.paths.HDF5Dtb + '/' + KinSource + '/' + SuffixHDF5
    )
    # Reading rates
    Systems[Identifier].Processes.read_rates(TTranVec=TTranVec, MicroRev=True)

### Plotting rates

In [13]:
iIdxVec = np.arange(5026, 5036) #np.random.randint(low=0, high=6114, size=10)

In [14]:
rates_type = RatesType(Systems['SurQCT'])
for iIdx in tqdm(iIdxVec):
    for i, TTran in enumerate(TTranVec):
        TTranStr = str(int(TTran))
        TrueData = Systems['SurQCT-ref'].Processes.Rates[RatesId][TTranStr][iIdx]
        PredData = Systems['SurQCT'].Processes.Rates[RatesId][TTranStr][iIdx]
        jIdxTrue = np.where(TrueData > MinRateValue)[0]
        jIdxPred = np.where(PredData > MinRateValue)[0]
        rates_type.plot_prediction3D(
            RatesType=RatesId,
            Path=out,
            iIdx=iIdx,
            TrueData=(jIdxTrue, TrueData[jIdxTrue]),
            PredData=(jIdxPred, PredData[jIdxPred])
        )

  0%|          | 0/10 [00:00<?, ?it/s]